In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import f_oneway

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

In [ ]:
df = pd.read_csv('../data/seasonal_agriculture_performance_dataset.csv')
print('Dataset loaded successfully!')
print('Dataset shape:', df.shape)
df.head()

In [ ]:
# Basic dataset information
print('Rows:', df.shape[0])
print('Columns:', df.shape[1])
print('\nColumn names:')
print(df.columns.tolist())
print('\nDataset information:')
df.info()
print('\nStatistical summary:')
df.describe()

In [ ]:
# Check missing values and duplicates
print('Missing values by column:')
print(df.isnull().sum())
print('\nDuplicate rows:', df.duplicated().sum())

In [ ]:
# Data cleaning
df = df.drop_duplicates().copy()
numeric_columns = df.select_dtypes(include=np.number).columns
for column in numeric_columns:
    df[column] = df[column].fillna(df[column].median())
categorical_columns = df.select_dtypes(include='object').columns
for column in categorical_columns:
    if df[column].isnull().sum() > 0:
        df[column] = df[column].fillna(df[column].mode()[0])
print('Shape after cleaning:', df.shape)
print('Total missing values after cleaning:', df.isnull().sum().sum())

In [ ]:
print('Seasons:')
print(df['Season'].value_counts())
print('\nCrops:')
print(df['Crop'].value_counts())
print('\nIrrigation methods:')
print(df['Irrigation_Method'].value_counts())
print('\nStates:')
print(df['State'].value_counts())

In [ ]:
season_analysis = df.groupby('Season').agg(
    Average_Yield=('Yield_Tonnes_Ha', 'mean'),
    Average_Profit=('Profit_INR', 'mean'),
    Average_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean'),
    Average_Disease_Pest_Risk=('Disease_Pest_Risk_pct', 'mean')
)
print(season_analysis)
highest_yield_season = season_analysis['Average_Yield'].idxmax()
highest_profit_season = season_analysis['Average_Profit'].idxmax()
print('\nHighest average yield season:', highest_yield_season)
print('Highest average profit season:', highest_profit_season)

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(season_analysis.index, season_analysis['Average_Yield'])
plt.title('Average Yield by Season')
plt.xlabel('Season')
plt.ylabel('Average Yield (Tonnes/Ha)')
plt.tight_layout()
plt.savefig('../visualizations/season_yield.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(season_analysis.index, season_analysis['Average_Profit'])
plt.title('Average Profit by Season')
plt.xlabel('Season')
plt.ylabel('Average Profit (INR)')
plt.tight_layout()
plt.savefig('../visualizations/season_profit.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
crop_analysis = df.groupby('Crop').agg(
    Average_Yield=('Yield_Tonnes_Ha', 'mean'),
    Average_Profit=('Profit_INR', 'mean')
)
print(crop_analysis)
best_yield_crop = crop_analysis['Average_Yield'].idxmax()
best_profit_crop = crop_analysis['Average_Profit'].idxmax()
print('\nHighest average yield crop:', best_yield_crop)
print('Highest average profit crop:', best_profit_crop)

In [ ]:
crop_profit = crop_analysis.sort_values('Average_Profit', ascending=False)
plt.figure(figsize=(10, 6))
plt.bar(crop_profit.index, crop_profit['Average_Profit'])
plt.title('Average Profit by Crop')
plt.xlabel('Crop')
plt.ylabel('Average Profit (INR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../visualizations/crop_profit.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
irrigation_analysis = df.groupby('Irrigation_Method').agg(
    Average_Yield=('Yield_Tonnes_Ha', 'mean'),
    Average_Water_Efficiency=('Water_Efficiency_t_per_1000m3', 'mean')
)
print(irrigation_analysis)
best_irrigation_yield = irrigation_analysis['Average_Yield'].idxmax()
best_water_method = irrigation_analysis['Average_Water_Efficiency'].idxmax()
print('\nHighest yield irrigation method:', best_irrigation_yield)
print('Highest water efficiency irrigation method:', best_water_method)

In [ ]:
irrigation_efficiency = irrigation_analysis.sort_values(
    'Average_Water_Efficiency', ascending=False
)
plt.figure(figsize=(9, 5))
plt.bar(irrigation_efficiency.index, irrigation_efficiency['Average_Water_Efficiency'])
plt.title('Water Efficiency by Irrigation Method')
plt.xlabel('Irrigation Method')
plt.ylabel('Water Efficiency (Tonnes/1000 m³)')
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('../visualizations/irrigation_efficiency.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
state_profit = df.groupby('State')['Profit_INR'].mean().sort_values(ascending=False)
print(state_profit)
highest_profit_state = state_profit.idxmax()
print('\nHighest average profit state:', highest_profit_state)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(state_profit.index, state_profit.values)
plt.title('Average Agricultural Profit by State')
plt.xlabel('State')
plt.ylabel('Average Profit (INR)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../visualizations/state_profit.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
correlation_columns = [
    'Rainfall_mm',
    'Avg_Temperature_C',
    'Humidity_pct',
    'Soil_Moisture_pct',
    'Nitrogen_kg_ha',
    'Fertilizer_kg_ha',
    'Seed_Quality_Score',
    'Yield_Tonnes_Ha'
]
correlation_matrix = df[correlation_columns].corr()
yield_correlation = correlation_matrix['Yield_Tonnes_Ha'].drop('Yield_Tonnes_Ha').sort_values(ascending=False)
print('Correlation matrix:')
print(correlation_matrix)
print('\nCorrelation with yield:')
print(yield_correlation)

In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(yield_correlation.index, yield_correlation.values)
plt.title('Correlation of Agricultural Factors with Yield')
plt.xlabel('Agricultural / Environmental Factor')
plt.ylabel('Correlation with Yield')
plt.xticks(rotation=45)
plt.axhline(y=0, linewidth=1)
plt.tight_layout()
plt.savefig('../visualizations/yield_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
yield_groups = [
    group['Yield_Tonnes_Ha'].dropna().values
    for _, group in df.groupby('Season')
]
yield_f_stat, yield_p_value = f_oneway(*yield_groups)
print('Yield ANOVA')
print('F-statistic:', yield_f_stat)
print('p-value:', yield_p_value)
if yield_p_value < 0.05:
    print('Seasonal yield differences are statistically significant.')
else:
    print('Seasonal yield differences are not statistically significant.')

In [ ]:
profit_groups = [
    group['Profit_INR'].dropna().values
    for _, group in df.groupby('Season')
]
profit_f_stat, profit_p_value = f_oneway(*profit_groups)
print('Profit ANOVA')
print('F-statistic:', profit_f_stat)
print('p-value:', profit_p_value)
if profit_p_value < 0.05:
    print('Seasonal profit differences are statistically significant.')
else:
    print('Seasonal profit differences are not statistically significant.')

In [ ]:
print('========== FINAL KEY FINDINGS ==========')
print('1. Highest average yield season:', highest_yield_season)
print('2. Highest average profit season:', highest_profit_season)
print('3. Highest average yield crop:', best_yield_crop)
print('4. Highest average profit crop:', best_profit_crop)
print('5. Highest yield irrigation method:', best_irrigation_yield)
print('6. Highest water efficiency irrigation method:', best_water_method)
print('7. Highest average profit state:', highest_profit_state)
print('8. Yield ANOVA p-value:', yield_p_value)
print('9. Profit ANOVA p-value:', profit_p_value)